In [ ]:
from dowhy import CausalModel
from experiments import df_twins_loaded, df_acs_loaded, df_walmart
from pprint import pprint
import pandas as pd

In [ ]:
def run_all_refutations(model, identified_estimand, estimate, df_name):
    refutation_results = []


    methods = [
        {"name": "random_common_cause", "label": "Add Random Common Cause", "kwargs": {"random_seed": 42}},
        {"name": "placebo_treatment_refuter", "label": "Placebo Treatment", "kwargs": {"placebo_type": "permute", "random_seed": 42}},
        {"name": "dummy_outcome_refuter", "label": "Dummy Outcome", "kwargs": {"random_seed": 42}},
        {"name": "data_subset_refuter", "label": "Data Subsets Validation", "kwargs": {"subset_fraction": 0.9, "random_seed": 42}},
        {"name": "bootstrap_refuter", "label": "Bootstrap Validation", "kwargs": {"num_simulations": 100, "random_seed": 42}},
        {"name": "add_unobserved_common_cause", "label": "Add Unobserved Common Cause", "kwargs":{"confounders_effect_on_treatment": "linear", "confounders_effect_on_outcome": "linear"}}
    ]
    for m in methods:
        try:
            print(f"Running {m['label']}...")
            res = model.refute_estimate(
                identified_estimand,
                estimate,
                method_name=m['name'],
                **m.get("kwargs", {})
            )
            if isinstance(res, list):
                res = res[0]
            # ---------------------------------------------
            print(f"ref results:{res.refutation_result}")
            p_val = res.refutation_result.get('p_value')

            status = "PASS" if (p_val is not None and p_val >= 0.05) else "FAIL"

            refutation_results.append({
                "Method": m['label'],
                "p-value": round(p_val, 4) if isinstance(p_val, (int, float)) else p_val,
                "Result": status
            })
        except Exception as e:
            print(e)
            refutation_results.append({
                "Method": m['label'],
                "p-value": "Error",
                "Result": f"Error: {str(e)}"
            })
    df = pd.DataFrame(refutation_results)
    df.to_csv(f"refutations_{df_name}.csv", index=False)
    return df


In [ ]:
model_twins=CausalModel(
        data = df_twins_loaded,
        treatment='treatment',
        outcome='outcome',
        common_causes=df_twins_loaded.columns.difference(["treatment", "outcome"]).tolist()
        )
identified_estimand_twins = model_twins.identify_effect(proceed_when_unidentifiable=True)
estimate_twins = model_twins.estimate_effect(identified_estimand_twins,method_name="backdoor.linear_regression")

model_acs=CausalModel(
        data = df_acs_loaded,
        treatment='treatment',
        outcome='outcome',
        common_causes=df_acs_loaded.columns.difference(["treatment", "outcome"]).tolist()
        )
identified_estimand_acs = model_acs.identify_effect(proceed_when_unidentifiable=True)
estimate_acs = model_acs.estimate_effect(identified_estimand_acs,method_name="backdoor.linear_regression")

model_walmart=CausalModel(
        data = df_walmart,
        treatment='treatment',
        outcome='outcome',
        common_causes=df_walmart.columns.difference(["treatment", "outcome"]).tolist()
        )
identified_estimand_walmart = model_walmart.identify_effect(proceed_when_unidentifiable=True)
estimate_walmart = model_walmart.estimate_effect(identified_estimand_walmart,method_name="backdoor.linear_regression")

In [ ]:
run_all_refutations(model_twins, identified_estimand_twins, estimate_twins, 'twins')

In [ ]:
run_all_refutations(model_acs, identified_estimand_acs, estimate_acs, 'acs')

In [ ]:
run_all_refutations(model_walmart, identified_estimand_walmart, estimate_walmart, 'walmart')

In [ ]:
refutation = model_twins.refute_estimate(
    identified_estimand_twins,
    estimate_twins,
    method_name="add_unobserved_common_cause",
    confounders_effect_on_treatment="linear",
    confounders_effect_on_outcome="linear"
)
print(refutation)

refutation = model_acs.refute_estimate(
    identified_estimand_acs,
    estimate_acs,
    method_name="add_unobserved_common_cause",
    confounders_effect_on_treatment="linear",
    confounders_effect_on_outcome="linear"
)
print(refutation)

refutation = model_walmart.refute_estimate(
    identified_estimand_walmart,
    estimate_walmart,
    method_name="add_unobserved_common_cause",
    confounders_effect_on_treatment="linear",
    confounders_effect_on_outcome="linear"
)
print(refutation)